In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

def plot_differences(file1, file2, output_path=None):
    arr1 = np.load(file1)
    arr2 = np.load(file2)
    t = arr1.shape[0]
    arr1 = arr1.reshape(t, -1)
    arr2 = arr2.reshape(t, -1)
    
    # Calculate differences
    diff = arr1 - arr2
    
    # For better visualization, limit extremely large values
    vmax = np.percentile(np.abs(diff), 95)  # Use 95th percentile for color scaling
    vmin = -vmax
    
    # Create a more detailed heatmap of differences
    plt.figure(figsize=(14, 10))
    
    # Use seaborn's heatmap which has better defaults for this type of visualization
    ax = sns.heatmap(
        diff, 
        cmap='coolwarm', 
        vmin=vmin, 
        vmax=vmax,
        center=0,
        cbar_kws={'label': 'Difference'},
        xticklabels=20,  # Show every 20th feature label
        yticklabels=50   # Show every 50th timestep label
    )
    
    plt.title(f'Differences between {os.path.basename(file1)} and {os.path.basename(file2)}', fontsize=14)
    plt.xlabel('Features', fontsize=12)
    plt.ylabel('Timesteps', fontsize=12)
    
    # Add annotations about statistics
    abs_diff = np.abs(diff)
    max_diff = np.max(abs_diff)
    mean_diff = np.mean(abs_diff)
    max_idx = np.unravel_index(np.argmax(abs_diff), abs_diff.shape)
    
    stats_text = f"Max diff: {max_diff:.4f} at {max_idx}\nMean diff: {mean_diff:.4f}"
    plt.figtext(0.02, 0.02, stats_text, fontsize=10, bbox=dict(facecolor='white', alpha=0.8))
    
    # Mark the location of maximum difference
    plt.plot(max_idx[1], max_idx[0], 'rx', markersize=8)
    
    plt.tight_layout()
    
    if output_path:
        plt.savefig(output_path, dpi=150, bbox_inches='tight')
        plt.close()
    else:
        plt.show()
        
    # Also create a simplified view showing just the magnitude of differences
    plt.figure(figsize=(14, 6))
    
    # Plot mean difference per feature
    feature_means = np.mean(abs_diff, axis=0)
    plt.subplot(1, 2, 1)
    plt.bar(range(len(feature_means)), feature_means)
    plt.title('Mean Diff by Feature')
    plt.xlabel('Feature Index')
    plt.ylabel('Mean Absolute Difference')
    
    # Plot mean difference per timestep
    timestep_means = np.mean(abs_diff, axis=1)
    plt.subplot(1, 2, 2)
    plt.plot(timestep_means)
    plt.title('Difference over Time')
    plt.xlabel('Timestep')
    plt.ylabel('Mean Absolute Difference')
    
    plt.tight_layout()
    
    if output_path:
        summary_path = output_path.replace('.png', '_summary.png')
        plt.savefig(summary_path, dpi=150, bbox_inches='tight')
        plt.close()
    else:
        plt.show()

In [ ]:
import os
import sys
import numpy as np
from pathlib import Path

def compare_npy_files(file1, file2, rtol=1e-05, atol=1e-08):
    arr1 = np.load(file1)
    arr2 = np.load(file2)
    equal = np.allclose(arr1, arr2, rtol=rtol, atol=atol)
    return equal

def cosine_similarity(vec1, vec2):
    norm1 = np.linalg.norm(vec1)
    norm2 = np.linalg.norm(vec2)
    if norm1 == 0 or norm2 == 0:
        return np.nan
    return np.dot(vec1, vec2) / (norm1 * norm2)

def compare_directories(dir1, dir2):
    discrepancies = []
    for root, _, files in os.walk(dir1):
        for file in files:
            if file.endswith('.npy'):
                rel_path = os.path.relpath(root, dir1)
                file1 = os.path.join(root, file)
                file2 = os.path.join(dir2, rel_path, f"M_{file}")
                if not os.path.exists(file2):
                    discrepancies.append(f"Missing file: {file2}")
                    continue

                # Load arrays once here to compute cosine similarity if needed.
                arr1 = np.load(file1)
                arr2 = np.load(file2)
                if not np.allclose(arr1, arr2, rtol=1e-05, atol=1e-08):
                    # Flatten arrays for cosine similarity calculation.
                    vec1 = arr1.ravel()
                    vec2 = arr2.ravel()
                    cos_sim = cosine_similarity(vec1, vec2)
                    discrepancies.append(f"Data differs: {Path(file1).stem} vs {Path(file2).stem} | {arr1.shape} vs {arr2.shape} | Cosine Similarity: {cos_sim:.4f}")
                    plot_save_path = os.path.join("kit_pose_data_diff", rel_path)
                    os.makedirs(plot_save_path, exist_ok=True)
                    #plot_differences(file1, file2, output_path=os.path.join(plot_save_path, f'diff_{file}'.replace('.npy', '.png')))
    return discrepancies

In [ ]:

directory1, directory2 = "kit_pose_data/KIT/KIT", "/lsdf/users/dschneider-kf3609/workspace/HumanML3D/joints_comp/KIT"
diffs = compare_directories(directory1, directory2)
if diffs:
    print("Differences found:")
    for d in diffs:
        print(d)
else:
    print("No differences found. The directories are identical (within tolerance).")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

def compare_pose_files(file1, file2, output_path=None):
    """Compare two pose files and visualize the differences"""
    # Load arrays
    arr1 = np.load(file1)
    arr2 = np.load(file2)
    
    # Print basic info
    print(f"File 1: {file1} - Shape: {arr1.shape}")
    print(f"File 2: {file2} - Shape: {arr2.shape}")
    
    # Check if shapes match
    if arr1.shape != arr2.shape:
        print(f"⚠️ Warning: Shapes do not match! {arr1.shape} vs {arr2.shape}")
        min_frames = min(arr1.shape[0], arr2.shape[0])
        arr1 = arr1[:min_frames]
        arr2 = arr2[:min_frames]
        print(f"Comparing first {min_frames} frames only")
    
    # Calculate differences and statistics
    diff = arr1 - arr2
    abs_diff = np.abs(diff)
    
    # Statistics
    max_diff = np.max(abs_diff)
    mean_diff = np.mean(abs_diff)
    max_idx = np.unravel_index(np.argmax(abs_diff), abs_diff.shape)
    cosine_sim = np.dot(arr1.flatten(), arr2.flatten()) / (
        np.linalg.norm(arr1.flatten()) * np.linalg.norm(arr2.flatten())
    )
    
    print(f"Max difference: {max_diff:.6f} at position {max_idx}")
    print(f"Mean difference: {mean_diff:.6f}")
    print(f"Cosine similarity: {cosine_sim:.6f} (1.0 = identical direction)")
    
    # Reshape for visualization if needed
    t = arr1.shape[0]
    vis_arr1 = arr1.reshape(t, -1)
    vis_arr2 = arr2.reshape(t, -1)
    vis_diff = diff.reshape(t, -1)
    
    # Visualize
    plt.figure(figsize=(15, 10))
    
    # Heatmap of differences
    vmax = np.percentile(np.abs(vis_diff), 95)
    vmin = -vmax
    ax = sns.heatmap(
        vis_diff,
        cmap='coolwarm',
        vmin=vmin,
        vmax=vmax,
        center=0,
        cbar_kws={'label': 'Difference'},
        xticklabels=20,
        yticklabels=50
    )
    
    plt.title(f'Differences between pose files', fontsize=14)
    plt.xlabel('Features', fontsize=12)
    plt.ylabel('Timesteps', fontsize=12)
    
    # Add annotations about statistics
    stats_text = f"Max diff: {max_diff:.4f} at {max_idx}\nMean diff: {mean_diff:.4f}\nCosine sim: {cosine_sim:.4f}"
    plt.figtext(0.02, 0.02, stats_text, fontsize=10, bbox=dict(facecolor='white', alpha=0.8))
    
    # Mark the location of maximum difference
    plt.plot(max_idx[1] if len(max_idx) > 1 else max_idx[0], 
             max_idx[0] if len(max_idx) > 1 else 0, 'rx', markersize=8)
    
    plt.tight_layout()
    
    if output_path:
        plt.savefig(output_path, dpi=150, bbox_inches='tight')
    else:
        plt.show()
    
    # Return a summary of the differences
    return {
        'max_diff': max_diff,
        'mean_diff': mean_diff,
        'cosine_sim': cosine_sim,
        'max_location': max_idx
    }

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
from matplotlib import gridspec

def animate_multiple_poses(
    pose_data_list,
    frames=200, 
    interval=50,
    elev=20, 
    azim=210,
    x_lim=[-1, 1],
    y_lim=[-1, 1],
    z_lim=[-1, 1],
    point_size=0.1,
    point_colors=None,
    transformation_matrices=None,
    titles=None
):
    """
    Generate an animation from multiple pose sequences displayed side by side.
    
    Parameters:
    -----------
    pose_data_list : list of numpy.ndarray
        List of pose data arrays, each with shape (frames, joints, 3)
    frames : int
        Number of frames to animate
    interval : int
        Interval between frames in milliseconds
    elev : float
        Elevation angle for 3D view
    azim : float
        Azimuth angle for 3D view
    x_lim, y_lim, z_lim : list
        Limits for each axis
    point_size : float
        Size of the points in the scatter plot
    point_colors : list of str, optional
        Colors for each pose sequence, defaults to ['r', 'b', 'g', ...]
    transformation_matrices : list of numpy.ndarray, optional
        List of 3x3 transformation matrices to apply to each pose sequence
    titles : list of str, optional
        Titles for each subplot
        
    Returns:
    --------
    HTML
        HTML representation of the animation
    """
    n_sequences = len(pose_data_list)
    
    # Default colors if not provided
    if point_colors is None:
        point_colors = ['r', 'b', 'g', 'c', 'm', 'y', 'k'] * (n_sequences // 7 + 1)
        point_colors = point_colors[:n_sequences]
    
    # Default transformation matrices if not provided
    if transformation_matrices is None:
        transformation_matrices = [None] * n_sequences
    elif len(transformation_matrices) < n_sequences:
        transformation_matrices += [None] * (n_sequences - len(transformation_matrices))
    
    # Default titles if not provided
    if titles is None:
        titles = [f"Sequence {i+1}" for i in range(n_sequences)]
    elif len(titles) < n_sequences:
        titles += [f"Sequence {i+1}" for i in range(len(titles), n_sequences)]
    
    # Apply transformations if provided
    transformed_poses = []
    for i, pose_data in enumerate(pose_data_list):
        if transformation_matrices[i] is not None:
            transformed_poses.append(np.dot(pose_data, transformation_matrices[i]))
        else:
            transformed_poses.append(pose_data)
    
    # Determine max frames across all sequences
    max_frames = min(frames, max(len(pose) for pose in transformed_poses))
    
    # Create figure with subplots
    fig = plt.figure(figsize=(5*n_sequences, 5))
    gs = gridspec.GridSpec(1, n_sequences)
    axes = [fig.add_subplot(gs[0, i], projection='3d') for i in range(n_sequences)]
    
    # Define visualization function
    def visualize_pose(frame):
        for i, ax in enumerate(axes):
            if frame < len(transformed_poses[i]):
                vertices = transformed_poses[i][frame]
                ax.cla()  # Clear the previous plot
                ax.scatter(vertices[:, 0], vertices[:, 1], vertices[:, 2], 
                          s=point_size, c=point_colors[i])
                ax.set_title(titles[i])
                ax.set_xlabel("X")
                ax.set_ylabel("Y")
                ax.set_zlabel("Z")
                ax.set_xlim(x_lim)
                ax.set_ylim(y_lim)
                ax.set_zlim(z_lim)
                ax.view_init(elev=elev, azim=azim)
    
    # Define update function for animation
    def update(frame):
        visualize_pose(frame)
        return axes
    
    # Create animation
    ani = FuncAnimation(fig, update, frames=max_frames, interval=interval)
    plt.close(fig)  # Prevents the static plot from showing
    
    # Return HTML representation
    return HTML(ani.to_jshtml())

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

def animate_overlayed_poses(
    pose_data1,
    pose_data2,
    frames=200, 
    interval=50,
    elev=20, 
    azim=210,
    x_lim=[-1, 1],
    y_lim=[-1, 1],
    z_lim=[-1, 1],
    point_size=0.5,
    point_colors=['r', 'b'],
    transformation_matrix1=None,
    transformation_matrix2=None,
    show_skeleton=False, # Set to True to draw lines
    skeleton_connections=None, # Provide skeleton connections if show_skeleton=True
    title="Overlayed Pose Animation",
    label1="Pose 1",
    label2="Pose 2"
):
    """
    Generate an animation overlaying two pose sequences in the same 3D plot.
    
    Parameters:
    -----------
    pose_data1, pose_data2 : numpy.ndarray
        Pose data arrays, each with shape (frames, joints, 3)
    frames : int
        Number of frames to animate
    interval : int
        Interval between frames in milliseconds
    elev : float
        Elevation angle for 3D view
    azim : float
        Azimuth angle for 3D view
    x_lim, y_lim, z_lim : list
        Limits for each axis
    point_size : float
        Size of the points in the scatter plot
    point_colors : list of str
        Colors for each pose sequence (should have length 2)
    transformation_matrix1, transformation_matrix2 : numpy.ndarray, optional
        3x3 transformation matrices to apply to each pose sequence
    show_skeleton : bool
        Whether to draw lines connecting joints based on skeleton_connections
    skeleton_connections : list of tuples, optional
        List of joint index pairs defining skeleton connections, e.g., [(0, 1), (1, 2), ...]
    title : str
        Title for the plot
    label1, label2 : str
        Labels for the legend corresponding to pose_data1 and pose_data2
        
    Returns:
    --------
    HTML
        HTML representation of the animation
    """
    
    # Apply transformations if provided
    if transformation_matrix1 is not None:
        pose_data1 = np.dot(pose_data1, transformation_matrix1)
    if transformation_matrix2 is not None:
        pose_data2 = np.dot(pose_data2, transformation_matrix2)
        
    # Determine max frames
    max_frames = min(frames, len(pose_data1), len(pose_data2))
    
    # Create figure and 3D axes
    fig = plt.figure(figsize=(7, 7))
    ax = fig.add_subplot(111, projection='3d')
    
    # Define update function for animation
    def update(frame):
        ax.cla()  # Clear the previous plot
        
        vertices1 = pose_data1[frame]
        vertices2 = pose_data2[frame]
        
        # Plot points for pose 1
        ax.scatter(vertices1[:, 0], vertices1[:, 1], vertices1[:, 2], 
                   s=point_size, c=point_colors[0], label=label1 if frame == 0 else "")
                   
        # Plot points for pose 2
        ax.scatter(vertices2[:, 0], vertices2[:, 1], vertices2[:, 2], 
                   s=point_size, c=point_colors[1], label=label2 if frame == 0 else "", alpha=0.7)

        # Draw skeleton lines if requested
        if show_skeleton and skeleton_connections is not None:
            # Skeleton for pose 1
            for connection in skeleton_connections:
                p1, p2 = connection
                if p1 < len(vertices1) and p2 < len(vertices1):
                    ax.plot([vertices1[p1, 0], vertices1[p2, 0]],
                            [vertices1[p1, 1], vertices1[p2, 1]],
                            [vertices1[p1, 2], vertices1[p2, 2]],
                            c=point_colors[0], alpha=0.6)
            # Skeleton for pose 2
            for connection in skeleton_connections:
                 p1, p2 = connection
                 if p1 < len(vertices2) and p2 < len(vertices2):
                    ax.plot([vertices2[p1, 0], vertices2[p2, 0]],
                            [vertices2[p1, 1], vertices2[p2, 1]],
                            [vertices2[p1, 2], vertices2[p2, 2]],
                            c=point_colors[1], alpha=0.4)

        # Set plot properties
        ax.set_title(f"{title} (Frame {frame})")
        ax.set_xlabel("X")
        ax.set_ylabel("Y")
        ax.set_zlabel("Z")
        ax.set_xlim(x_lim)
        ax.set_ylim(y_lim)
        ax.set_zlim(z_lim)
        ax.view_init(elev=elev, azim=azim)
        if frame == 0: # Add legend only once
             ax.legend()

    # Create animation
    ani = FuncAnimation(fig, update, frames=max_frames, interval=interval)
    plt.close(fig)  # Prevents the static plot from showing
    
    # Return HTML representation
    return HTML(ani.to_jshtml())

In [ ]:
# Load two pose sequences
path1 = "/home/rdueger/HumanML3D/try_pose_data/EyesJapanDataset/Eyes_Japan_Dataset/ichige/pain-01-headache-ichige_poses.npy"
path2 = "/lsdf/users/dschneider-kf3609/workspace/HumanML3D/joints_comp/EyesJapanDataset/ichige/M_pain-01-headache-ichige_poses.npy"  # Replace with another pose file

pose_data1 = np.load(path1, allow_pickle=True)
pose_data2 = np.load(path2, allow_pickle=True)

# Apply transformation matrix to both sequences
trans_matrix = np.array([[1.0, 0.0, 0.0], [0.0, 0.0, 1.0], [0.0, 1.0, 0.0]])

# Generate and display animations side by side
animation = animate_multiple_poses(
    pose_data_list=[pose_data1, pose_data2],
    transformation_matrices=[trans_matrix, trans_matrix],
    titles=["Comparison Pose", "Original Pose"],
    point_colors=["r", "b"],
    point_size=0.5,
    frames=200,
    interval=50,
    elev=90,
    azim=360,
)
animation

In [ ]:
# Generate and display the overlayed animation
animation = animate_overlayed_poses(
    pose_data1=pose_data1,
    pose_data2=pose_data2,
    transformation_matrix1=trans_matrix,
    transformation_matrix2=trans_matrix,
    frames=min(200, len(pose_data1), len(pose_data2)), # Ensure frames don't exceed data length
    interval=50,
    point_size=1.0, # Slightly larger points might help visibility
    point_colors=['red', 'blue'],
    elev=20, # Top-down view might reveal planar differences
    azim=210,  # Align X-axis horizontally
    label1="Comparison Pose",
    label2="Original Pose"
)
animation

In [ ]:
compare_pose_files(path1, path2)

In [ ]:
# Check for numerical closeness
are_close = np.allclose(pose_data1, pose_data2, rtol=1e-04, atol=1e-06) # Default tolerances
print(f"Are arrays numerically close (default tolerances)? {are_close}")

# Check with stricter tolerances if needed
are_very_close = np.allclose(pose_data1, pose_data2, rtol=1e-07, atol=1e-10)
print(f"Are arrays numerically close (stricter tolerances)? {are_very_close}")

# Check for exact equality (usually False for floats)
are_equal = np.array_equal(pose_data1, pose_data2)
print(f"Are arrays exactly equal? {are_equal}")

In [ ]:
diff = pose_data1 - pose_data2

if len(pose_data1.shape) == 3:
    num_joints = pose_data1.shape[1]
    
    # Mean absolute difference per joint
    mean_diff_per_joint = np.mean(np.abs(diff), axis=(0, 2))
    # Mean absolute difference per axis (X, Y, Z)
    mean_diff_per_axis = np.mean(np.abs(diff), axis=(0, 1))
    
    plt.figure(figsize=(12, 5))
    
    plt.subplot(1, 2, 1)
    plt.bar(range(num_joints), mean_diff_per_joint)
    plt.title('Mean Absolute Difference per Joint')
    plt.xlabel('Joint Index')
    plt.ylabel('Mean Abs Diff')
    
    plt.subplot(1, 2, 2)
    plt.bar(['X', 'Y', 'Z'], mean_diff_per_axis)
    plt.title('Mean Absolute Difference per Axis')
    plt.xlabel('Axis')
    plt.ylabel('Mean Abs Diff')
    
    plt.tight_layout()
    plt.show()
    
    print("Mean difference per joint:", mean_diff_per_joint)
    print("Mean difference per axis:", mean_diff_per_axis)
else:
    print("Array shape not (frames, joints, 3), skipping per-joint/axis analysis.")